In [ ]:
# stdlib pathlib (filesystem paths)
from pathlib import Path

# src/utils/pathing.py
from src.utils.pathing import ensure_repo_root_on_sys_path  # src/utils/pathing.py

ensure_repo_root_on_sys_path(Path.cwd())

# 35. Neural Networks: Recurrent Neural Networks (RNNs) and LSTM

## Algorithm Category
**Type**: Neural Networks - Deep Learning  
**Complexity**: High  
**Use Case**: Sequence modeling, time series prediction, natural language processing

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand RNNs and their architecture
- Implement RNNs and LSTMs using PyTorch
- Understand vanishing/exploding gradient problems
- Apply LSTMs to sequence prediction tasks
- Visualize hidden states and predictions
- Compare RNN vs LSTM performance

## Historical Context

RNNs were developed in the 1980s:
- Rumelhart, D.E., et al. (1986): "Learning representations by back-propagating errors"
- LSTM introduced by Hochreiter & Schmidhuber (1997)
- Foundation for modern sequence modeling

**Key Papers/References:**
- Hochreiter, S., & Schmidhuber, J. (1997). "Long short-term memory"
- Rumelhart, D.E., et al. (1986). "Learning representations by back-propagating errors"

## When to Use RNNs and LSTMs

RNNs/LSTMs are appropriate when:
- Working with sequential data
- Time series prediction
- Natural language processing
- Speech recognition
- When order matters in data
- Need to remember past information

## Theory & Mechanics

### Mathematical Foundation

**RNN Hidden State:**
$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$

**RNN Output:**
$$y_t = W_{hy} h_t + b_y$$

**LSTM Cell:**
- **Forget gate**: $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$
- **Input gate**: $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$
- **Cell state**: $C_t = f_t * C_{t-1} + i_t * \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$
- **Output gate**: $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$
- **Hidden state**: $h_t = o_t * \tanh(C_t)$

### Key Components

1. **RNN**
   - Processes sequences step by step
   - Maintains hidden state
   - Shares weights across time steps
   - Suffers from vanishing gradients

2. **LSTM (Long Short-Term Memory)**
   - Solves vanishing gradient problem
   - Uses gates to control information flow
   - Can remember long-term dependencies
   - More complex than RNN

3. **Gates in LSTM**
   - **Forget gate**: What to forget
   - **Input gate**: What to remember
   - **Output gate**: What to output

### How It Works

1. **Initialize**: Hidden state $h_0$
2. **Process sequence**: For each time step:
   - Compute hidden state (RNN) or cell/hidden (LSTM)
   - Generate output
3. **Backpropagation**: Through time (BPTT)
4. **Update weights**: Gradient descent

### Key Hyperparameters

- **hidden_size**: Number of hidden units
- **num_layers**: Number of RNN/LSTM layers
- **sequence_length**: Length of input sequences
- **dropout**: Regularization rate
- **bidirectional**: Process in both directions

### Advantages

- Handles variable-length sequences
- Can model temporal dependencies
- Shares parameters across time
- Good for sequential data
- LSTM solves vanishing gradients

### Limitations

- Slow training (sequential processing)
- Hard to parallelize
- May forget long-term dependencies (RNN)
- Computationally expensive
- Requires careful initialization


## Implementation

Let's implement RNNs and LSTMs for sequence prediction.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# PyTorch: Deep learning framework
import torch  # PyTorch core library (tensors, automatic differentiation)
import torch.nn as nn  # Neural network layers and modules
import torch.optim as optim  # Optimization algorithms (Adam, SGD, etc.)
from torch.utils.data import DataLoader, TensorDataset  # Data loading utilities

# Scikit-learn: Machine learning utilities
from sklearn.preprocessing import MinMaxScaler  # Feature scaling (normalize to [0, 1])
from sklearn.metrics import mean_squared_error  # Evaluation metric (MSE)

# ============================================
# GPU DETECTION: Using CUDA for Faster Training
# ============================================

# Check if CUDA (GPU) is available
# RNNs/LSTMs can be slow - GPU acceleration helps significantly
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# torch.cuda.is_available(): Returns True if GPU is available
# If GPU available: device = 'cuda' (use GPU)
# If no GPU: device = 'cpu' (use CPU - slower but still works)
print(f"Using device: {device}")  # Display which device we're using

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# GENERATING SYNTHETIC TIME SERIES DATA
# ============================================

# RNNs/LSTMs are designed for sequential data (time series, sequences)
# We'll create a sine wave time series to demonstrate sequence prediction

def generate_sine_wave(n_samples=1000, noise_level=0.1):
    """
    Generate sine wave time series with noise.
    
    Args:
        n_samples: Number of time steps to generate
        noise_level: Standard deviation of random noise
    
    Returns:
        Array of time series values
    """
    # Create time points from 0 to 4π (two full sine wave cycles)
    t = np.linspace(0, 4*np.pi, n_samples)
    # np.linspace(): Creates evenly spaced values
    # 0 to 4π: Two complete sine wave cycles
    
    # Generate sine wave with added noise
    data = np.sin(t) + noise_level * np.random.randn(n_samples)
    # np.sin(t): Pure sine wave (smooth, predictable pattern)
    # noise_level * np.random.randn(n_samples): Random noise (makes it realistic)
    # np.random.randn(): Standard normal distribution (mean=0, std=1)
    # Adding noise simulates real-world data (not perfectly smooth)
    
    return data

# ============================================
# GENERATING AND PREPARING DATA
# ============================================

# Generate time series data
data = generate_sine_wave(n_samples=1000, noise_level=0.1)
# n_samples=1000: 1000 time steps
# noise_level=0.1: Small amount of noise (10% of signal amplitude)

# Reshape to 2D array (required by scaler)
data = data.reshape(-1, 1)
# Original: (1000,) - 1D array
# Reshaped: (1000, 1) - 2D array (samples × features)
# Scikit-learn expects 2D arrays

# ============================================
# NORMALIZATION: Scaling to [0, 1]
# ============================================

# Normalize data to [0, 1] range
# This helps neural networks train faster and more stably
scaler = MinMaxScaler()
# MinMaxScaler: Scales data to [0, 1] range
# Formula: (x - min) / (max - min)

data_scaled = scaler.fit_transform(data)
# fit_transform(): Learn min/max from data and apply scaling
# fit: Learn min and max values
# transform: Apply scaling: (x - min) / (max - min)
# Result: All values between 0 and 1

# ============================================
# DISPLAYING DATA INFORMATION
# ============================================

print(f"Time series data shape: {data_scaled.shape}")  # Should be (1000, 1)
print(f"Data range: [{data_scaled.min():.3f}, {data_scaled.max():.3f}]")  # Should be [0.0, 1.0]

# ============================================
# VISUALIZING TIME SERIES
# ============================================

# Plot first 200 time steps to see the pattern
plt.figure(figsize=(12, 4))  # Figure size: 12×4 inches
plt.plot(data_scaled[:200])  # Plot first 200 time steps
# Shows the sine wave pattern with noise
# RNN/LSTM should learn to predict the next value in this sequence

plt.xlabel('Time Step')  # X-axis: time step number
plt.ylabel('Value')  # Y-axis: normalized value (0-1)
plt.title('Sample Time Series (First 200 Steps)')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid for easier reading
plt.tight_layout()  # Adjust layout
plt.show()  # Display the plot

# Interpretation:
# - Sine wave pattern: Predictable, repeating pattern
# - Noise: Random variations (makes prediction harder)
# - RNN/LSTM should learn: "Given past values, predict next value"
# - This is a sequence-to-one prediction task


In [ ]:
# Prepare sequences for RNN/LSTM
def create_sequences(data, seq_length=10):
    """Create sequences for time series prediction"""
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 20
X, y = create_sequences(data_scaled, seq_length)

# Split data
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Training sequences: {X_train.shape}")
print(f"Test sequences: {X_test.shape}")

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train).to(device)
X_test_tensor = torch.FloatTensor(X_test).to(device)
y_train_tensor = torch.FloatTensor(y_train).to(device)
y_test_tensor = torch.FloatTensor(y_test).to(device)


In [ ]:
# ============================================
# DEFINING RNN MODEL: Basic Recurrent Neural Network
# ============================================

# SimpleRNN: Basic RNN for sequence prediction
# RNNs process sequences step-by-step, maintaining a hidden state
class SimpleRNN(nn.Module):
    """
    Simple RNN for time series prediction.
    
    Architecture:
    - RNN layer: Processes sequence, maintains hidden state
    - Fully connected: Maps hidden state to output
    """
    
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, output_size=1):
        """
        Initialize RNN model.
        
        Args:
            input_size: Number of features per time step (1 for univariate time series)
            hidden_size: Number of hidden units (RNN memory capacity)
            num_layers: Number of RNN layers (stacked RNNs)
            output_size: Number of output values (1 for predicting next value)
        """
        super(SimpleRNN, self).__init__()
        # super() calls parent class (nn.Module) constructor
        
        # Store hyperparameters
        self.hidden_size = hidden_size  # Size of hidden state
        self.num_layers = num_layers  # Number of RNN layers
        
        # RNN layer: Processes sequences
        # nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        # - input_size: Features per time step (1 for our data)
        # - hidden_size: Number of hidden units (32)
        # - num_layers: Number of stacked RNN layers (1)
        # - batch_first=True: Input shape is (batch, seq, features) instead of (seq, batch, features)
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        # RNN processes sequence step-by-step, updating hidden state at each step
        
        # Fully connected layer: Maps hidden state to output
        # nn.Linear(hidden_size, output_size)
        # - hidden_size: Input size (32)
        # - output_size: Output size (1 for predicting next value)
        self.fc = nn.Linear(hidden_size, output_size)
        # Takes final hidden state and produces prediction
    
    def forward(self, x):
        """
        Forward pass through RNN.
        
        Args:
            x: Input tensor (batch_size, seq_length, input_size)
        
        Returns:
            Output tensor (batch_size, output_size)
        """
        # Initialize hidden state to zeros
        # Shape: (num_layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        # x.size(0): Batch size
        # .to(x.device): Move to same device as input (GPU or CPU)
        # Hidden state starts at zero (no memory initially)
        
        # RNN forward pass
        # Processes entire sequence, updating hidden state at each step
        out, hn = self.rnn(x, h0)
        # x: Input sequence (batch, seq_length, input_size)
        # h0: Initial hidden state
        # Returns:
        #   - out: Output at each time step (batch, seq_length, hidden_size)
        #   - hn: Final hidden state (num_layers, batch, hidden_size)
        
        # Take last output (final time step)
        # out[:, -1, :]: Last time step for each sequence in batch
        out = self.fc(out[:, -1, :])
        # out[:, -1, :]: Shape (batch, hidden_size) - last hidden state
        # self.fc(): Maps to output (batch, output_size)
        
        return out  # Return prediction

# ============================================
# DEFINING LSTM MODEL: Long Short-Term Memory
# ============================================

# SimpleLSTM: LSTM for sequence prediction
# LSTMs solve the vanishing gradient problem of RNNs
# Can remember long-term dependencies better than RNNs
class SimpleLSTM(nn.Module):
    """
    Simple LSTM for time series prediction.
    
    Architecture:
    - LSTM layer: Processes sequence with gates (forget, input, output)
    - Fully connected: Maps hidden state to output
    """
    
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, output_size=1):
        """
        Initialize LSTM model.
        
        Args:
            input_size: Number of features per time step (1)
            hidden_size: Number of hidden units (32)
            num_layers: Number of LSTM layers (1)
            output_size: Number of output values (1)
        """
        super(SimpleLSTM, self).__init__()
        
        # Store hyperparameters
        self.hidden_size = hidden_size  # Size of hidden state
        self.num_layers = num_layers  # Number of LSTM layers
        
        # LSTM layer: Processes sequences with gates
        # nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        # - input_size: Features per time step (1)
        # - hidden_size: Number of hidden units (32)
        # - num_layers: Number of stacked LSTM layers (1)
        # - batch_first=True: Input shape is (batch, seq, features)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        # LSTM has gates:
        # - Forget gate: What to forget from previous state
        # - Input gate: What new information to remember
        # - Output gate: What to output
        # This allows LSTM to remember long-term dependencies!
    
    def forward(self, x):
        """
        Forward pass through LSTM.
        
        Args:
            x: Input tensor (batch_size, seq_length, input_size)
        
        Returns:
            Output tensor (batch_size, output_size)
        """
        # Initialize hidden and cell states to zeros
        # LSTM has two states (RNN only has one):
        # - Hidden state (h): Short-term memory
        # - Cell state (c): Long-term memory
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        # Both states start at zero (no memory initially)
        
        # LSTM forward pass
        # Processes entire sequence, updating both states at each step
        out, (hn, cn) = self.lstm(x, (h0, c0))
        # x: Input sequence (batch, seq_length, input_size)
        # (h0, c0): Initial hidden and cell states
        # Returns:
        #   - out: Output at each time step (batch, seq_length, hidden_size)
        #   - (hn, cn): Final hidden and cell states
        
        # Take last output (final time step)
        out = self.fc(out[:, -1, :])
        # out[:, -1, :]: Last hidden state (batch, hidden_size)
        # self.fc(): Maps to output (batch, output_size)
        
        return out  # Return prediction

print("RNN and LSTM models defined!")  # Confirm models are ready

# Key Differences:
# - RNN: Simple, one hidden state, suffers from vanishing gradients
# - LSTM: More complex, two states (hidden + cell), can remember long-term dependencies
# - LSTM typically performs better on longer sequences


## Training LSTM

Let's train the LSTM model.


In [ ]:
# Train LSTM
model_lstm = SimpleLSTM(input_size=1, hidden_size=32, num_layers=1, output_size=1).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)

num_epochs = 50
train_losses = []

for epoch in range(num_epochs):
    model_lstm.train()
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model_lstm(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}: Loss = {loss.item():.6f}")

print("\nTraining complete!")


## Evaluation and Comparison

Let's compare RNN and LSTM performance.


In [ ]:
# Train RNN for comparison
model_rnn = SimpleRNN(input_size=1, hidden_size=32, num_layers=1, output_size=1).to(device)
optimizer_rnn = optim.Adam(model_rnn.parameters(), lr=0.001)

rnn_losses = []
for epoch in range(num_epochs):
    model_rnn.train()
    optimizer_rnn.zero_grad()
    outputs = model_rnn(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer_rnn.step()
    rnn_losses.append(loss.item())

# Evaluate both models
model_lstm.eval()
model_rnn.eval()

with torch.no_grad():
    lstm_pred = model_lstm(X_test_tensor).cpu().numpy()
    rnn_pred = model_rnn(X_test_tensor).cpu().numpy()

# Calculate MSE
lstm_mse = mean_squared_error(y_test, lstm_pred)
rnn_mse = mean_squared_error(y_test, rnn_pred)

print("Model Comparison:")
print(f"  LSTM Test MSE: {lstm_mse:.6f}")
print(f"  RNN Test MSE: {rnn_mse:.6f}")

# Plot training losses
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='LSTM', alpha=0.7)
plt.plot(rnn_losses, label='RNN', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss: LSTM vs RNN')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Visualization

Let's visualize the predictions.


In [ ]:
# Visualize predictions
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# LSTM predictions
axes[0].plot(y_test[:200], label='True', alpha=0.7, linewidth=2)
axes[0].plot(lstm_pred[:200], label='LSTM Prediction', alpha=0.7, linestyle='--')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Value')
axes[0].set_title('LSTM Predictions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RNN predictions
axes[1].plot(y_test[:200], label='True', alpha=0.7, linewidth=2)
axes[1].plot(rnn_pred[:200], label='RNN Prediction', alpha=0.7, linestyle='--')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Value')
axes[1].set_title('RNN Predictions')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Validation & Testing

Let's validate the models.


In [ ]:
# Assertions
assert lstm_mse < 0.1, "LSTM should learn the pattern"
assert rnn_mse < 0.2, "RNN should learn the pattern"
print("\n✓ Validation checks passed")

print("\nNote: LSTM typically performs better on longer sequences")
print("due to its ability to remember long-term dependencies.")


## Summary & Key Takeaways

### Key Concepts Learned

1. **RNN Basics**
   - Processes sequences step by step
   - Maintains hidden state
   - Shares weights across time
   - Suffers from vanishing gradients

2. **LSTM (Long Short-Term Memory)**
   - Solves vanishing gradient problem
   - Uses gates to control information
   - Can remember long-term dependencies
   - More complex but more powerful

3. **Key Components**
   - **Hidden state**: Carries information
   - **Gates**: Control information flow (LSTM)
   - **Cell state**: Long-term memory (LSTM)
   - **Sequence length**: How far back to look

4. **Applications**
   - Time series prediction
   - Natural language processing
   - Speech recognition
   - Sequence-to-sequence tasks

### When to Use RNNs and LSTMs

✅ **Good for:**
- Sequential data
- Time series prediction
- Natural language processing
- When order matters
- Variable-length sequences
- Need to remember past context

❌ **Not ideal for:**
- Tabular data (use MLPs)
- Image data (use CNNs)
- Very long sequences (use Transformers)
- When parallelization is critical
- Real-time applications (can be slow)

### Next Steps

- Explore **GRU (Gated Recurrent Unit)** as LSTM alternative
- Try **Bidirectional RNNs/LSTMs**
- Apply to **NLP tasks** (text generation, sentiment)
- Use **Attention mechanisms** for better performance
- Experiment with **Transformer models** for long sequences
